# CSA — real-data XE learning pilot
Add BOTH Datasets: `csa-real-v126-lemma-fixed` and `csa-semantic-banks-v130` (owner ducanh2403). Enable Internet + GPU; GPU0 only, no API Secret.
Fresh self-written CSA;256TRAIN×5captions,64val;15small-subsetepochs=600updates;effectivebatch32(micro4×8),FP32;3epochwarmup8e-6→1e-4.
No CLIP/ROI recache, full COCO, RL or official CIDEr/CHAIR. This pilot tests learning, not novelty/grounding accuracy.
Hard worker cap90minutes (not ETA). Save & Run All. Send reviewZIP; preserve checkpoint.bin and both input Datasets.


In [ ]:
import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile,shutil
from pathlib import Path
PREPARED_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-real-v126-lemma-fixed')
BANK_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-semantic-banks-v130')
REPORTED_GPU_HOURS=19
OUTPUT=Path('/kaggle/working/csa_real_xe_pilot_256')
COMMIT='19fed3eea2cd864415bd0503d38bdac4566d1871'
HASHES={'run_xe_pilot.py': '4f2c92b4a5a56545863618212ec04daef35ce3da58b6de5a088b10f41ad85ba5', 'pilot_data.py': '134cc6de37a6a2afd6d9d242ea74edddde08eb3f2fd90771e34c779ae104f820', 'real_bank_data.py': '00e7110105ec98a5ef41969f9e51074815a42a9b69deac90b2614e9679d2bcf9', 'model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09', 'REAL_BANK_INPUT_PACKAGE.json': 'c60902e04ea15cf90f652e384937bb317e288753f13b8ede6ab5ef89f8d97129', 'SAVE_BANK_V130_PACKAGE.json': '826ba36628d6587860e2d548e950ea953bd4bbb8c0a8dc64393089dbc1e058e1', 'REAL_XE_PILOT_PROTOCOL.md': 'df1f2e0c6a7b0a183767b7a861203128e9529280053780829dff07ef1ad50e47'}
if OUTPUT.exists():raise RuntimeError('Preserve existing output/checkpoint; use a fresh session.')
if not PREPARED_ROOT.is_dir() or not BANK_ROOT.is_dir():raise FileNotFoundError('Add both Dataset inputs first.')
if shutil.disk_usage('/kaggle/working').free<3*1024**3:raise RuntimeError('Need3GiB free for atomic checkpoint writes.')
OUTPUT.mkdir(parents=True)
SOURCE=Path(tempfile.mkdtemp(prefix='csa-real-xe-'))
LOG=OUTPUT/'notebook.log'
print('Source:',COMMIT,'Pilot:15small-subsetepochs/600updates/GPU0/hardcap90min',flush=True)
for name,h in HASHES.items():
    payload=urllib.request.urlopen(f'https://raw.githubusercontent.com/Supzxjee/Image_Captioning/{COMMIT}/experiments/csa_paper/{name}',timeout=90).read()
    if hashlib.sha256(payload).hexdigest()!=h:raise ValueError('Source SHA differs: '+name)
    (SOURCE/name).write_bytes(payload)
(OUTPUT/'source_receipt.json').write_text(json.dumps(dict(commit=COMMIT,sha256=HASHES,reported_gpu_hours=REPORTED_GPU_HOURS),indent=2),encoding='utf-8')


In [ ]:
def run(args, timeout=1800):
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('Pilot active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('Pilot wall cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
complete=False
try:
    # Existing Torch/NumPy only: no encoder/model downloads or pip installs.
    run([sys.executable,'-u',SOURCE/'run_xe_pilot.py','--prepared-root',PREPARED_ROOT,
         '--bank-root',BANK_ROOT,'--output',OUTPUT],timeout=5400)
    complete=True
except Exception:
    (OUTPUT/'failure.txt').write_text(traceback.format_exc(),encoding='utf-8')
    raise
finally:
    review=Path(str(OUTPUT)+'_review.zip')
    with zipfile.ZipFile(review,'w',zipfile.ZIP_DEFLATED) as z:
        for path in sorted(OUTPUT.iterdir()):
            if path.is_file() and path.suffix in ('.json','.txt','.log','.md'):z.write(path,path.name)
    print('SEND REVIEW:',review,flush=True)
    print('KEEP CHECKPOINT:',OUTPUT/'checkpoint.bin',flush=True)
    print('COMPLETE' if complete else 'INCOMPLETE; report completed epochs/updates only.',flush=True)
